<td>
<a href="https://colab.research.google.com/github/raoulg/MADS-DAV/blob/main/notebooks/lesson6/06.5-your-own-vectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
</td>


# 6.5 Your own vectors — what is one row, when a row is "haha"?

6.4's flower cache worked because one row was one flower: a clean photo, a clean label,
encoded once. Chat data offers no such luck. A large share of messages are five words or
fewer, and a three-word message embeds to something almost content-free — not because the
encoder is bad, but because there is barely anything there to encode.

This notebook stages that failure, fixes it, and checks the fix actually worked — on the
IRC showcase first, where six prolific authors give the checks something to check against,
then on your own chat, with the same functions.

> Needs `torch`, `sentence-transformers` and `transformers`: `uv sync --extra huggingface`
> once, locally.


In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from notebooktester import param

from goad_toolkit.visualizer import HeatmapPlot, PlotSettings, ProjectionPlot
from scripts.pipelines import build_irc_pipeline
from scripts.sessionize import fit_session_threshold, merge_messages, sessionize
from wa_analyzer.data import load_own_chat, load_showcase

EMBEDDER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


## B1 · Stage the blob

Six of #ubuntu-uk's most active posters, one row per message, embedded exactly as they
were typed. If an individual message carried much signal about who wrote it, a 2D
projection should already show six clusters — the same claim 6.4 checked and confirmed
for flower photos.


In [ ]:
irc = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))
uk = irc[(irc.channel == "#ubuntu-uk") & (~irc.is_action)].copy()
uk["timestamp"] = (
    pd.to_datetime(uk["date"]) + pd.to_timedelta(uk["hh"], unit="h") + pd.to_timedelta(uk["mm"], unit="m")
)
uk = uk.sort_values(["author", "timestamp"]).reset_index(drop=True)

TOP_AUTHORS = uk.author.value_counts().head(6).index.tolist()
print(f"{len(uk):,} messages, median {uk.message.str.split().str.len().median():.0f} words, "
      f"top authors: {TOP_AUTHORS}")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(42)
BLOB_SAMPLE = param(1500, test=150)
sample = uk[uk.author.isin(TOP_AUTHORS)].sample(n=BLOB_SAMPLE, random_state=42)

X = EMBEDDER.encode(sample.message.tolist(), batch_size=64, show_progress_bar=False)
coords = PCA(n_components=2).fit_transform(X)

settings = PlotSettings(
    figsize=(7, 6),
    title="1,500 individual messages, six authors — embedded one at a time",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=coords, labels=sample.author.to_numpy(),
                                        palette="tab10", s=10)

print(f"silhouette by author: {silhouette_score(X, sample.author):.3f}  (0 = no structure)")


A blob, and a silhouette within rounding error of zero — precisely what 6.4's flowers
picture looked like *before* PCA, except this is *after* embedding. The encoder did not
fail; "thanks!" and "lol" simply do not carry six people's worth of identity in three
words. **This is a stage-2 problem — what counts as one row — not a stage-4 problem with
the model.**


## B2 and B3 · Three ways to define one row

Merge consecutive messages from the same author into a unit, three different ways:

1. **Fixed window** — every 15 messages from one author, whatever they happen to be about.
2. **Per-author-per-day** — everything one person said on one calendar day.
3. **Time-gap sessions** — a new unit whenever the gap since that person's last message is
   longer than a threshold. That threshold should not be guessed.

Lesson 4 fit an exponential to the gaps between messages, restricted to gaps under an hour
— the within-burst regime — and read the fit as evidence of *when* people talk in bursts
rather than at a steady rate. The same fit, on this data, gives the number sessionization
actually needs: the gap length beyond which a continuation looks more like a coincidence
than a reply.


In [ ]:
threshold = fit_session_threshold(uk["timestamp"], uk["author"])
print(f"session threshold: {threshold:.0f}s ({threshold / 60:.1f} minutes)")

uk["session_id"] = sessionize(uk, "timestamp", "author", threshold)
subset = uk[uk.author.isin(TOP_AUTHORS)].copy()

K = 15
subset["window_id"] = subset.groupby("author").cumcount() // K
subset["day"] = subset["timestamp"].dt.date

fixed_units = merge_messages(subset, ["author", "window_id"])
daily_units = merge_messages(subset, ["author", "day"])
session_units = merge_messages(subset, ["author", "session_id"])

for name, units in [("fixed-window", fixed_units), ("per-day", daily_units), ("session", session_units)]:
    words = units.message.str.split().str.len()
    print(f"{name:>12s}: {len(units):>6,} units, median {words.median():>4.0f} words each")


~15 minutes, read off the fit rather than picked. Three defensible ways to answer "what
is one row" — and the next section checks whether the answer actually matters, instead of
just admiring three different numbers.


## Checking the merge, not just admiring it

The original plan here was an external check: `jkkummerfeld/irc-disentanglement`
publishes human-annotated reply links for IRC. It does not cover this course's slice —
its annotations are drawn from the `#ubuntu` channel, not `#ubuntu-uk` or `#ubuntu-nl`,
and this repo does not even vendor `#ubuntu`'s raw text (only its hourly counts, deliberately,
because the full log is ~400 MB). Verified, and dropped, rather than quietly skipped.

A self-supervised check stands in instead, and it needs no external annotation: split each
unit in half by time, embed the two halves separately, and ask whether a nearest-neighbour
search on cosine similarity finds the true other half more often than chance. A merge
strategy that produces topically coherent units should pass this easily; one that produces
noise should not do better than chance.


In [ ]:
def self_retrieval_accuracy(frame: pd.DataFrame, group_cols: list[str], min_messages: int = 6,
                            n_eval: int = 400, seed: int = 42) -> tuple[float, int]:
    """Split each group's messages in half by time, embed both halves, and check whether
    cosine similarity finds the true other half more often than chance."""
    sizes = frame.groupby(group_cols).size()
    keys = sizes[sizes >= min_messages].index.tolist()
    rng = np.random.default_rng(seed)
    if len(keys) > n_eval:
        keys = [keys[i] for i in rng.choice(len(keys), size=n_eval, replace=False)]

    first, second = [], []
    grouped = frame.sort_values("timestamp").groupby(group_cols)
    for key in keys:
        grp = grouped.get_group(key)
        half = len(grp) // 2
        first.append(" ".join(grp.message.iloc[:half]))
        second.append(" ".join(grp.message.iloc[half:]))

    A = EMBEDDER.encode(first, batch_size=64, show_progress_bar=False)
    B = EMBEDDER.encode(second, batch_size=64, show_progress_bar=False)
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)
    nearest = (A @ B.T).argmax(axis=1)
    accuracy = float((nearest == np.arange(len(keys))).mean())
    return accuracy, len(keys)


Single raw messages cannot be split in half, so their baseline is measured slightly
differently: one message from early in a burst against one from later in the same burst,
rather than a text split down the middle. Everything else uses the function above directly.


In [ ]:
def raw_message_baseline(frame: pd.DataFrame, group_cols: list[str], min_messages: int = 6,
                         n_eval: int = 400, seed: int = 42) -> tuple[float, int]:
    """Same check, but each 'half' is a single message rather than several joined together."""
    sizes = frame.groupby(group_cols).size()
    keys = sizes[sizes >= min_messages].index.tolist()
    rng = np.random.default_rng(seed)
    if len(keys) > n_eval:
        keys = [keys[i] for i in rng.choice(len(keys), size=n_eval, replace=False)]

    first, second = [], []
    grouped = frame.sort_values("timestamp").groupby(group_cols)
    for key in keys:
        grp = grouped.get_group(key)
        half = len(grp) // 2
        first.append(grp.message.iloc[0])
        second.append(grp.message.iloc[half])

    A = EMBEDDER.encode(first, batch_size=64, show_progress_bar=False)
    B = EMBEDDER.encode(second, batch_size=64, show_progress_bar=False)
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)
    nearest = (A @ B.T).argmax(axis=1)
    accuracy = float((nearest == np.arange(len(keys))).mean())
    return accuracy, len(keys)


In [ ]:
N_EVAL = param(400, test=40)

results = {"single message": raw_message_baseline(subset, ["author", "session_id"], n_eval=N_EVAL)}
results["per-day"] = self_retrieval_accuracy(subset, ["author", "day"], n_eval=N_EVAL)
results["fixed-window"] = self_retrieval_accuracy(subset, ["author", "window_id"], n_eval=N_EVAL)
results["session"] = self_retrieval_accuracy(subset, ["author", "session_id"], n_eval=N_EVAL)

print(f"{'strategy':>14s}  {'accuracy':>9s}  {'n pairs':>8s}   vs. chance")
for name, (acc, n) in results.items():
    chance = 1 / n if n else float("nan")
    print(f"{name:>14s}  {acc:>8.1%}   {n:>7,}   {acc / chance:>5.0f}x")


Every merge strategy beats a single message by a wide margin, and **time-gap
sessionization wins outright** — the units it produces are the ones most reliably matched
back to their other half by meaning alone. Fixed windows chop mid-thought at message 15
regardless of what is happening; per-day units dilute one real conversation among however
many separate ones that person had that day. Getting the boundary right, the way B3 fit
it, is not a nicety — it is the difference between a usable unit and a diluted one.


## B5 · Cosine similarity on merged units

Two things a merged, embedded corpus is good for: finding what is similar to one thing,
and asking whether "similar" lines up with something you can already check.


In [ ]:
units = session_units[session_units.n_messages >= 5].reset_index(drop=True)
vectors = EMBEDDER.encode(units.message.tolist(), batch_size=64, show_progress_bar=False)
normed = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

rng = np.random.default_rng(7)
query = rng.integers(0, len(units))
similarity = normed @ normed[query]
ranked = np.argsort(similarity)
nearest, farthest = ranked[-2], ranked[0]  # -1 is the query itself

for label, i in [("QUERY", query), ("NEAREST", nearest), ("FARTHEST", farthest)]:
    print(f"{label} ({units.author.iloc[i]}, cosine={similarity[i]:.2f}):")
    print(f"  {units.message.iloc[i][:200]}")
    print()


The nearest session is about the same kind of thing the query was — both are
about squeezing more out of limited memory, one for swap and one for a GPU — and
the farthest is a completely different conversation, about routing audio over a
network. Cosine similarity on merged units is retrieving by topic, which is
exactly what it was asked to do and exactly what individual messages could not
support.


In [ ]:
centroids = pd.DataFrame(
    {author: normed[units.author.to_numpy() == author].mean(axis=0) for author in TOP_AUTHORS}
).T
centroids = centroids.div(np.linalg.norm(centroids, axis=1), axis=0)
author_similarity = pd.DataFrame(
    centroids.to_numpy() @ centroids.to_numpy().T, index=TOP_AUTHORS, columns=TOP_AUTHORS
)

settings = PlotSettings(figsize=(6, 5), title="Author centroids, cosine similarity", xlabel="", ylabel="")
fig, ax = HeatmapPlot(settings).plot(data=author_similarity, fmt=".2f")


Every pair sits at 0.82–0.94 — high, and barely discriminating between six different
people. That is not a bug in the method; it is what a **semantic** embedding is supposed
to do. Everyone in this channel is discussing Ubuntu support, so everyone's centroid points
roughly the same direction. Lesson 5's dialect fingerprints (`nose` vs `whom nose`, ☺ vs
☺) separated these same eight people cleanly — but that signal is *stylistic*, encoded in
punctuation and word choice, not in *meaning*. An embedding model measures the thing it was
trained to measure, and that is content, not habit. Reach for the right tool: trigram
counts for style, embeddings for topic.


## B6 · Race the baseline against the embedding

6.2 built a baseline that costs nothing to fit: character-trigram counts. On the exact
same merged units used above, does a trained-nowhere sentence embedding actually beat it
at the one task it should have an edge on — telling six people apart from what they wrote?


In [ ]:
PER_AUTHOR_CAP = param(600, test=80)
balanced = units.groupby("author", group_keys=False).apply(
    lambda g: g.sample(min(len(g), PER_AUTHOR_CAP), random_state=42), include_groups=False
).join(units[["author"]])
train, test = train_test_split(balanced, test_size=0.3, stratify=balanced.author, random_state=42)

trigrams = CountVectorizer(analyzer="char", ngram_range=(3, 3), min_df=2)
Xtr_cv = trigrams.fit_transform(train.message)
Xte_cv = trigrams.transform(test.message)
acc_trigram = accuracy_score(test.author, LogisticRegression(max_iter=1000).fit(Xtr_cv, train.author).predict(Xte_cv))

Xtr_emb = EMBEDDER.encode(train.message.tolist(), batch_size=64, show_progress_bar=False)
Xte_emb = EMBEDDER.encode(test.message.tolist(), batch_size=64, show_progress_bar=False)
acc_embed = accuracy_score(test.author, LogisticRegression(max_iter=1000).fit(Xtr_emb, train.author).predict(Xte_emb))

baseline = test.author.value_counts(normalize=True).max()
print(f"majority-class baseline : {baseline:.1%}")
print(f"trigram counts + logreg : {acc_trigram:.1%}")
print(f"sentence embedding + logreg : {acc_embed:.1%}")


The cheap, hand-designed, un-pretrained trigram vector **wins by a wide margin** — both
comfortably beat guessing the majority author, but counting characters identifies these six
people far better than a general-purpose sentence embedding does. This is B5's finding
again from a different angle: the embedding is good at *what was said*, and identity here
lives in *how*. On six labelled classes and a few thousand rows, the interpretable model is
not a fallback for when you cannot afford the fancy one — on this task it is the better
tool, and you would only know that by racing them.


## B7 · A sentiment score, as a closer

One more transfer-learned model, briefly: a sentiment classifier, run on the same merged
session units. No projection, no PCA — just a label and a confidence per unit, aggregated
per author.


In [ ]:
from transformers import pipeline as hf_pipeline

sentiment_model = hf_pipeline(  # ty: ignore[no-matching-overload]
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    top_k=None, truncation=True, max_length=512,
)

SENTIMENT_SAMPLE = param(60, test=10)
sample_units = units.sample(n=SENTIMENT_SAMPLE, random_state=0)
scores = []
for _, row in sample_units.iterrows():
    result = {item["label"]: item["score"] for item in sentiment_model(row.message[:512])[0]}
    scores.append({"author": row.author, **result})

sentiment = pd.DataFrame(scores)
sentiment.groupby("author")[["positive", "neutral", "negative"]].mean().round(2)


A table, not a claim — sixty units is nowhere near enough to say anything about any one
person's disposition, and a support channel's "negative" score is mostly *other people's
problems*, not the speaker's mood. Treat this the way 5.4 treated a p-value on its own:
a number, not yet a finding, until it has a mechanism and a replication behind it.


## Your turn

Same functions, your own chat. `fit_session_threshold` and `sessionize` do not know or
care that the data used to be IRC — they take a timestamp column and an author column,
which your own export has too.


In [ ]:
own = load_own_chat()

if own is not None:
    own_threshold = fit_session_threshold(own["timestamp"], own["author"])
    print(f"your session threshold: {own_threshold:.0f}s ({own_threshold / 60:.1f} minutes)")

    own = own.sort_values(["author", "timestamp"]).reset_index(drop=True)
    own["session_id"] = sessionize(own, "timestamp", "author", own_threshold)
    own_units = merge_messages(own, ["author", "session_id"])
    checkable = own_units[own_units.n_messages >= 5]

    print(f"{len(own_units)} session units, {len(checkable)} with 5+ messages")
    if len(checkable) >= 20:
        acc, n = self_retrieval_accuracy(own, ["author", "session_id"], min_messages=5, n_eval=200)
        print(f"self-retrieval accuracy on your sessions: {acc:.1%} (n={n}, chance={1 / n:.1%})")
    else:
        print("Not enough multi-message sessions yet for the retrieval check -- "
              "the merge itself is still the useful part.")
else:
    print("No own chat loaded -- see the message above for how to get one.")


## The bridge to `tensordtype 2`

The ML course meets this same shape from the opposite direction: a document too *long*
for a model's context window gets split into chunks, each chunk becomes a vector, and the
chunks get aggregated back into one representation — `(C, D) → (D)`. This notebook merged
messages too *short* to mean anything into units, embedded each unit, same tensor shape,
same aggregation step. Long-split-down or short-merged-up, the destination is identical:
one row, one meaningful vector, and a padding problem waiting on the other side of it.
Same problem, opposite cause — and the cheapest possible head start on the ML course's
`tensordtype 2`.
